In [ ]:
import pandas as pd

data_dir = "Data/aptos2019-blindness-detection"

train_csv = "E:\\AI_Project\\Data\\aptos2019-blindness-detection\\train.csv"
train_images_dir = "E:\\AI_Project\\Data\\aptos2019-blindness-detection\\train_images"

df = pd.read_csv(train_csv)

df["filename"] = df["id_code"] + ".png"
df["label"] = df["diagnosis"].astype(str)

print("Total images:", len(df))
df.head()


Total images: 3662


,id_code,diagnosis,filename,label
0,000c1434d8d7,2,000c1434d8d7.png,2
1,001639a390f0,4,001639a390f0.png,4
2,0024cdab0c1e,1,0024cdab0c1e.png,1
3,002c21358ce6,0,002c21358ce6.png,0
4,005b95c28852,0,005b95c28852.png,0


In [13]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras import layers, models
from sklearn.model_selection import train_test_split

# ==========================
# 1. Paths
# ==========================
train_csv = "E:\\AI_Project\\Data\\aptos2019-blindness-detection\\train.csv"
train_images_dir = "E:\\AI_Project\\Data\\aptos2019-blindness-detection\\train_images"

# ==========================
# 2. Load CSV
# ==========================
df = pd.read_csv(train_csv)

df["filename"] = df["id_code"] + ".png"
df["label"] = df["diagnosis"].astype(str)

print("Total images:", len(df))

# ==========================
# 3. Train / Validation Split
# ==========================
train_df, val_df = train_test_split(
    df,
    test_size=0.15,
    stratify=df["label"],
    random_state=42
)

# ==========================
# 4. Image Preprocessing
# ==========================
IMG_SIZE = 224
BATCH_SIZE = 32

train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=15,
    zoom_range=0.1,
    horizontal_flip=True
)

val_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_dataframe(
    train_df,
    directory=train_images_dir,
    x_col="filename",
    y_col="label",
    target_size=(IMG_SIZE, IMG_SIZE),
    class_mode="categorical",
    batch_size=BATCH_SIZE
)

val_generator = val_datagen.flow_from_dataframe(
    val_df,
    directory=train_images_dir,
    x_col="filename",
    y_col="label",
    target_size=(IMG_SIZE, IMG_SIZE),
    class_mode="categorical",
    batch_size=BATCH_SIZE
)

# ==========================
# 5. CNN Model Creation
# ==========================
model = models.Sequential([

    layers.Conv2D(32, (3,3), activation='relu', input_shape=(IMG_SIZE, IMG_SIZE, 3)),
    layers.MaxPooling2D(2,2),

    layers.Conv2D(64, (3,3), activation='relu'),
    layers.MaxPooling2D(2,2),

    layers.Conv2D(128, (3,3), activation='relu'),
    layers.MaxPooling2D(2,2),

    layers.Flatten(),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.5),

    layers.Dense(5, activation='softmax')  # 5 DR classes
])

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

# ==========================
# 6. Train Model
# ==========================
EPOCHS = 15

history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=EPOCHS
)

# ==========================
# 7. Save Model
# ==========================
model.save("aptos_dr_model.keras")
print("Model saved successfully!")


Total images: 3662
Found 3112 validated image filenames belonging to 5 classes.
Found 550 validated image filenames belonging to 5 classes.



Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d (Conv2D)             (None, 222, 222, 32)      896       
                                                                 
 max_pooling2d (MaxPooling2  (None, 111, 111, 32)      0         
 D)                                                              
                                                                 
 conv2d_1 (Conv2D)           (None, 109, 109, 64)      18496     
                                                                 
 max_pooling2d_1 (MaxPoolin  (None, 54, 54, 64)        0         
 g2D)                                                            
                                                                 
 conv2d_2 (Conv2D)           (None, 52, 52, 1

In [4]:
import tensorflow as tf
print(tf.config.list_physical_devices('GPU'))

ModuleNotFoundError: No module named 'tensorflow.python.util.lazy_loader'